# LLM-Augmented AlphaGen — Ablation Analysis

Compares 3 conditions × 2 seeds = 6 pools:

| Condition | Idea-agent warm-start | Judge post-filter |
|---|---|---|
| `A_vanilla` | ✗ | ✗ |
| `B_warm` | ✓ | ✗ |
| `C_warm_judge` | ✓ | ✓ |

Inputs:
- `data/factors/{A,B,C}_*_seed{42,1337}_pool.json` — final pool state + val/test IC
- `out/tensorboard/` — per-rollout `test/ic_mean` learning curves (parsed via tbparse)

Outputs (the table + 2 plots you take into the internship deck):
1. Summary table: test IC, RankIC, top-5 single IC, pool diversity, pool size
2. Learning curve plot: `test/ic_mean` vs steps, mean ± seed-range band per condition
3. Per-condition factor breakdown


In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "external" / "alphagen"))

# ──────────────────────────────────────────────────────────────
# CONFIG — change DATA_SOURCE to switch the analysis target.
# ──────────────────────────────────────────────────────────────
DATA_SOURCE = "cn"        # "cn" for CSI300 experiment, "crypto" for the crypto smoke test
SEEDS = [42, 1337]
SUFFIX = "_cn" if DATA_SOURCE == "cn" else ""
CONDITIONS = [f"A_vanilla{SUFFIX}", f"B_warm{SUFFIX}", f"C_warm_judge{SUFFIX}"]
FACT_DIR = ROOT / "data" / "factors"
TB_DIR = ROOT / "out" / "tensorboard"

from alphagen.config import OPERATORS
from alphagen.data.parser import ExpressionParser
from src.factor_mining._calc_factory import build_calculators

def pool_path(cond, seed):
    return FACT_DIR / f"{cond}_seed{seed}_pool.json"

def load_pool(cond, seed):
    p = pool_path(cond, seed)
    if not p.exists():
        return None
    return json.loads(p.read_text())

pools = {(c, s): load_pool(c, s) for c in CONDITIONS for s in SEEDS}
for k, v in pools.items():
    print(f"{k}: {'OK' if v else 'MISSING'} ({pool_path(*k).name})")


## 1. Headline summary table

In [ ]:
calcs = build_calculators(
    data_source=DATA_SOURCE,
    data_config_path="../config/data_config.yaml",
    splits_to_load=("train", "test"),
)
train_calc = calcs["train"]
test_calc = calcs["test"]
parser = ExpressionParser(
    operators=OPERATORS,
    ignore_case=False,
    time_deltas_need_suffix=True,
    non_positive_time_deltas_allowed=False,
    feature_need_dollar_sign=True,
)

def pool_metrics(pool: dict) -> dict:
    exprs = [parser.parse(e) for e in pool["exprs"]]
    single_ics_test = [float(test_calc.calc_single_IC_ret(e)) for e in exprs]
    single_ics_train = [float(train_calc.calc_single_IC_ret(e)) for e in exprs]
    abs_test = sorted([abs(x) for x in single_ics_test], reverse=True)
    if len(exprs) < 2:
        diversity = float("nan")
    else:
        mat = np.stack([train_calc.evaluate_alpha(e).cpu().numpy().ravel() for e in exprs])
        mat = np.nan_to_num(mat, nan=0.0)
        corr = np.corrcoef(mat)
        iu = np.triu_indices_from(corr, k=1)
        diversity = float(np.nanmean(np.abs(corr[iu])))
    return {
        "pool_size": len(exprs),
        "val_ic": pool.get("val_ic"),
        "val_ric": pool.get("val_ric"),
        "test_ic": pool.get("test_ic"),
        "test_ric": pool.get("test_ric"),
        "top5_test_abs_single_ic": float(np.mean(abs_test[:5])) if abs_test else float("nan"),
        "mean_train_abs_single_ic": float(np.mean(np.abs(single_ics_train))) if single_ics_train else float("nan"),
        "pool_mean_abs_corr": diversity,
    }

rows = []
for (c, s), p in pools.items():
    if p is None:
        continue
    m = pool_metrics(p)
    rows.append({"condition": c, "seed": s, **m})
raw = pd.DataFrame(rows)
raw


In [ ]:
# Aggregate across seeds: mean and seed-range (min, max)
agg_cols = [c for c in raw.columns if c not in ('condition', 'seed')]
agg = raw.groupby('condition')[agg_cols].agg(['mean', 'min', 'max'])
agg = agg.reindex(CONDITIONS)
agg.round(4)

## 2. Learning curves from tensorboard

In [ ]:
# Requires: pip install tbparse
try:
    from tbparse import SummaryReader
    HAVE_TB = True
except ImportError:
    print('tbparse not installed. `pip install tbparse` to enable learning-curve plots.')
    HAVE_TB = False

def load_curves(tag: str) -> pd.DataFrame:
    runs = list(TB_DIR.glob(f'{tag}_*'))
    if not runs:
        return pd.DataFrame()
    run_dir = sorted(runs)[-1]
    reader = SummaryReader(str(run_dir), pivot=False)
    df = reader.scalars
    return df[df['tag'].isin(['test/ic_mean', 'test/rank_ic_mean', 'pool/best_ic_ret'])].copy()

if HAVE_TB:
    curves = {(c, s): load_curves(f'{c}_seed{s}') for c in [f'A_vanilla{SUFFIX}', f'B_warm{SUFFIX}'] for s in SEEDS}
    for k, v in curves.items():
        print(f'{k}: {len(v)} rows, tags={sorted(v["tag"].unique()) if len(v) else []}')

In [ ]:
if HAVE_TB:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    palette = {f'A_vanilla{SUFFIX}': 'tab:gray', f'B_warm{SUFFIX}': 'tab:blue'}
    for ax, metric, title in zip(axes, ['test/ic_mean', 'pool/best_ic_ret'], ['Test IC (ensemble)', 'Best train IC (pool)']):
        for cond in [f'A_vanilla{SUFFIX}', f'B_warm{SUFFIX}']:
            series_list = []
            for s in SEEDS:
                df = curves.get((cond, s), pd.DataFrame())
                if df.empty: continue
                m = df[df['tag'] == metric].sort_values('step')
                if not len(m): continue
                series_list.append(pd.Series(m['value'].values, index=m['step'].values))
            if not series_list: continue
            aligned = pd.concat(series_list, axis=1).ffill()
            mean = aligned.mean(axis=1)
            lo, hi = aligned.min(axis=1), aligned.max(axis=1)
            ax.plot(mean.index, mean.values, color=palette[cond], label=cond, linewidth=2)
            ax.fill_between(mean.index, lo.values, hi.values, alpha=0.2, color=palette[cond])
        ax.set_xlabel('PPO timestep')
        ax.set_ylabel(metric)
        ax.set_title(title)
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / 'out' / 'learning_curves.png', dpi=120)
    plt.show()

## 3. Inspect C: which factors did the judge drop?

In [ ]:
for s in SEEDS:
    p = pools.get((CONDITIONS[2], s))
    if p is None or 'filter' not in p:
        continue
    info = p['filter']
    print(f'seed={s}  kept={info["n_kept"]}/{info["n_input"]}  threshold={info["threshold"]}')
    print('-- dropped --')
    judges = info.get('judge_results', [])
    for r in judges:
        if not r['kept']:
            print(f'  score={r["score"]:.2f}  {r["expr"]}')
            if r.get('nl'): print(f'             NL: {r["nl"][:120]}')
    print()

## 4. Warm-seed quality (sanity check on idea-agent output)

In [ ]:
for s in SEEDS:
    p = FACT_DIR / f'warm_seeds_cn_seed{s}.json' if DATA_SOURCE == 'cn' else f'warm_seeds_seed{s}.json'
    if not p.exists():
        print(f'seed={s}: no warm-seeds file')
        continue
    payload = json.loads(p.read_text())
    print(f'seed={s}: generated={payload["n_generate"]}, parsed={payload["n_parsed"]}, '
          f'passed_ic={payload["n_scored"]}, selected={payload["n_selected"]}')
    for sd in payload['seeds']:
        print(f'  IC={sd["train_ic"]:+.4f}  [{sd["family"]:>12s}]  {sd["expr"]}')
        print(f'             {sd["hypothesis"]}')

## 5. Final write-up table

Copy these numbers directly into the internship slide.

In [ ]:
summary = pd.DataFrame({
    'condition': CONDITIONS,
})
for col in ['test_ic', 'test_ric', 'top5_test_abs_single_ic', 'pool_mean_abs_corr', 'pool_size']:
    means = raw.groupby('condition')[col].mean().reindex(CONDITIONS)
    mins  = raw.groupby('condition')[col].min().reindex(CONDITIONS)
    maxs  = raw.groupby('condition')[col].max().reindex(CONDITIONS)
    summary[col] = [f'{m:.3f}  [{lo:.3f}, {hi:.3f}]' for m, lo, hi in zip(means, mins, maxs)]
summary.set_index('condition')